## GraphFlow

Graphflow provides a way of defining structured execution and provide control on how agents to accomplish a task.

### DiGraph Builder
utitlity to build DiGraph which can then be used in Graphflow. Using diagraph we can easily build agent interactions including

1. Sequential Chain
2. Parallel 

In [1]:
import os
import re
import asyncio
from typing import List, Sequence
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
import warnings
warnings.filterwarnings("ignore")

load_dotenv()
api_key = os.getenv("AICREDITS_API_KEY")
base_url = os.getenv("BASE_URL")

if not api_key:
    raise RuntimeError("Please set the AICREDITS_API_KEY environment variable.")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini", api_key=api_key, base_url=base_url, temperature=0
)

### Sequential Flow

In [2]:
writer = AssistantAgent(
    name = 'Writer',
    model_client=model_client,
    description='A writer agent that generates text based on user input.',
    system_message="You are a creative writer. Please write a story based on the user's input."
)

reviewer = AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [3]:
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph = builder.build()

In [4]:
graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

```text
DiGraph(nodes={
    'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 
    'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)
```

In [5]:
team = GraphFlow([writer,reviewer], graph)
stream = team.run_stream(task="Write a good poem about India in less than 30 words")

async for event in stream:
    print(event)

id='bfe45062-a4b6-4f88-86d1-e75d5cf072bd' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 10, 20, 7, 39, 27, 414847, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words' type='TextMessage'
id='23e2fc3c-cf6a-4779-87a8-eca958cbb503' source='Writer' models_usage=RequestUsage(prompt_tokens=40, completion_tokens=42) metadata={} created_at=datetime.datetime(2026, 10, 20, 7, 39, 32, 19107, tzinfo=datetime.timezone.utc) content="In vibrant hues, the land unfolds,  \nAncient tales in whispers told.  \nRivers dance, mountains rise,  \nUnity in diversity, beneath vast skies.  \nIndia's heart, where spirit flies." type='TextMessage'
id='c1ca6033-e38b-4779-94b5-b6f45610b831' source='Reviewer' models_usage=RequestUsage(prompt_tokens=88, completion_tokens=101) metadata={} created_at=datetime.datetime(2026, 10, 20, 7, 39, 33, 902040, tzinfo=datetime.timezone.utc) content='This poem beautifully captures the essence of India in just a

In [ ]:
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")

# Create a directed graph with fan-out flow A -> (B, C).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
# A --> B & A --> C
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)

id='669ecf6f-dcb2-4551-bf64-48633d2942f5' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 10, 20, 7, 40, 21, 45961, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='3792d281-3fef-41f1-936e-4eadaf894bbe' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=432) metadata={} created_at=datetime.datetime(2026, 10, 20, 7, 40, 26, 788325, tzinfo=datetime.timezone.utc) content="Once upon a time in a quaint little village, there lived a fluffy orange cat named Marmalade. Marmalade was not just any cat; he was known throughout the village for his adventurous spirit and his insatiable curiosity. Every morning, as the sun peeked over the rooftops, Marmalade would stretch his paws, yawn widely, and set off on his daily explorations.\n\nOne sunny day, while wandering through the village square, Marmalade noticed a commotion near the fountain. A group of children had gathered, their eyes wide with excit

In [7]:
graph

DiGraph(nodes={'A': DiGraphNode(name='A', edges=[DiGraphEdge(target='B', condition=None, condition_function=None, activation_group='B', activation_condition='all'), DiGraphEdge(target='C', condition=None, condition_function=None, activation_group='C', activation_condition='all')], activation='all'), 'B': DiGraphNode(name='B', edges=[], activation='all'), 'C': DiGraphNode(name='C', edges=[], activation='all')}, default_start_node=None)

```text
DiGraph(nodes={
    'A': DiGraphNode(name='A', edges=[
        DiGraphEdge(target='B', condition=None, condition_function=None, activation_group='B', activation_condition='all'), 
        DiGraphEdge(target='C', condition=None, condition_function=None, activation_group='C', activation_condition='all')], activation='all'), 
    'B': DiGraphNode(name='B', edges=[], activation='all'), 
    'C': DiGraphNode(name='C', edges=[], activation='all')}, default_start_node=None)
```

In [8]:
agent_d = AssistantAgent("D", model_client=model_client, system_message="You are a helpful assistant.")
agent_e = AssistantAgent("E", model_client=model_client, system_message="Translate input to Chinese.")
agent_f = AssistantAgent("F", model_client=model_client, system_message="Translate input to Japanese.")
agent_g = AssistantAgent("G", model_client=model_client, system_message="Summarize.")


builder = DiGraphBuilder()
builder.add_node(agent_d).add_node(agent_e).add_node(agent_f).add_node(agent_g)
builder.add_edge(agent_d, agent_e).add_edge(agent_d, agent_f)
builder.add_edge(agent_e, agent_g).add_edge(agent_f, agent_g)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_d, agent_e, agent_f, agent_g],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)

id='b21ebf18-83a8-43b5-9bba-91a2d29cd37d' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 10, 20, 8, 5, 59, 224297, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='c93f2784-15a4-4600-b0d3-743d650c3dcc' source='D' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=584) metadata={} created_at=datetime.datetime(2026, 10, 20, 8, 6, 5, 167591, tzinfo=datetime.timezone.utc) content='Once upon a time in a quaint little village, there lived a fluffy orange cat named Marmalade. Marmalade was not just any cat; he was known throughout the village for his adventurous spirit and his insatiable curiosity. Every morning, as the sun peeked over the rooftops, Marmalade would stretch his paws, yawn widely, and set off on his daily explorations.\n\nOne sunny day, while wandering through the village square, Marmalade noticed a commotion near the fountain. A group of children had gathered, their laughter ringing throu

In [9]:
agent_h = AssistantAgent(
    "H",
    model_client=model_client,
    system_message="Detect if the input is in Chinese. If it is, say 'yes', else say 'no', and nothing else.",
)
agent_i = AssistantAgent("I", model_client=model_client, system_message="Translate input to English.")
agent_j = AssistantAgent("J", model_client=model_client, system_message="Translate input to Chinese.")

# Create a directed graph with conditional branching flow H -> I ("yes"), H -> J (otherwise).
builder = DiGraphBuilder()
builder.add_node(agent_h).add_node(agent_i).add_node(agent_j)
# Create conditions as callables that check the message content.
builder.add_edge(agent_h, agent_i, condition=lambda msg: "yes" in msg.to_model_text())
builder.add_edge(agent_h, agent_j, condition=lambda msg: "yes" not in msg.to_model_text())
graph = builder.build()

team = GraphFlow(
    participants=[agent_h, agent_i, agent_j],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="AutoGen is a framework for building AI agents."):
    print(event)

id='3fae3cfb-8715-48fb-9a59-8b248d4bf30d' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 10, 20, 8, 15, 32, 659502, tzinfo=datetime.timezone.utc) content='AutoGen is a framework for building AI agents.' type='TextMessage'
id='39b61fce-8bf8-4631-92ee-6db58f7aee07' source='H' models_usage=RequestUsage(prompt_tokens=47, completion_tokens=1) metadata={} created_at=datetime.datetime(2026, 10, 20, 8, 15, 35, 671201, tzinfo=datetime.timezone.utc) content='no' type='TextMessage'
id='58140df5-05ca-4022-bb02-4802423972f6' source='J' models_usage=RequestUsage(prompt_tokens=33, completion_tokens=13) metadata={} created_at=datetime.datetime(2026, 10, 20, 8, 15, 36, 909590, tzinfo=datetime.timezone.utc) content='AutoGen是一个用于构建AI代理的框架。' type='TextMessage'
messages=[TextMessage(id='3fae3cfb-8715-48fb-9a59-8b248d4bf30d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 10, 20, 8, 15, 32, 659502, tzinfo=datetime.timezone.utc), content='Aut